In [0]:
s3_ext_location = 's3://awae-yellow-taxi-case/'

from pyspark.sql import functions as F

In [0]:
# 1. Load baseline integrated movement data from Silver Layer (Delta)
df_silver = spark.read.format("delta").load(f"{s3_ext_location}/silver/taxi_tripdata_silver")

# 2. Load Dimension Lookup tables from Silver Layer (Parquet)
df_dim_provider = spark.read.format("parquet").load(f"{s3_ext_location}/silver/dim_tpep_provider")
df_dim_ratecode = spark.read.format("parquet").load(f"{s3_ext_location}/silver/dim_ratecode")
df_dim_payment = spark.read.format("parquet").load(f"{s3_ext_location}/silver/dim_payment_type")

In [0]:
# 3. Apply business rule filtering on the fact stream
df_fact_filtered = df_silver.filter("total_amount > 0 AND passenger_count > 0")

# 4. Perform Left Joins to enrich metrics stream with textual metadata descriptions
df_enriched = df_fact_filtered \
    .join(df_dim_provider, on="VendorID", how="left") \
    .join(df_dim_ratecode, on="RatecodeID", how="left") \
    .join(df_dim_payment, on="payment_type", how="left")

In [0]:
# 5. Extract temporal components and drop original numeric keys to preserve only descriptions
df_gold_curated = df_enriched \
    .withColumn("pickup_year", F.year(F.col("tpep_pickup_datetime"))) \
    .withColumn("pickup_month", F.date_format(F.col("tpep_pickup_datetime"), "MM")) \
    .withColumn("pickup_day", F.dayofmonth(F.col("tpep_pickup_datetime"))) \
    .withColumn("pickup_hour", F.hour(F.col("tpep_pickup_datetime"))) \
    .select(
        F.col("VendorID"),
        F.col("VendorDesc"),
        F.col("passenger_count"),
        F.col("total_amount"),
        F.col("tpep_pickup_datetime"),
        F.col("tpep_dropoff_datetime"),
        F.col("taxi_type"),
        F.col("RatecodeDesc"),
        F.col("payment_desc"),
        F.col("pickup_year"),
        F.col("pickup_month"),
        F.col("pickup_day"),
        F.col("pickup_hour")
    ) \
    .filter("pickup_year IS NOT NULL AND pickup_month IS NOT NULL")

In [0]:
# 6. Save the final curated star-schema fact table to Gold layer in high-performance Parquet format
path_gold = f"{s3_ext_location}/gold/taxi_trips_curated/"
print(f"Writing curated granular enriched fact dataset to Gold layer (Parquet): {path_gold}")

df_gold_curated.write.mode("overwrite").parquet(path_gold)
print("✅ Gold analytical fact table successfully generated and ready for analysis!")

In [0]:
display(df_gold_curated.limit(10))

In [0]:
display(df_gold_curated.select("taxi_type").groupBy("taxi_type").count())